# Wan 2.2 Kaggle: ammonium chloride salt analysis (persistent worker)

Persistent-worker pilot based on the original six-segment salt-analysis prompts. This version uses the actual salt-analysis reference image, not the previous redox-titration image. It generates one short independent shot per major procedure segment so each clip can reset to a verified apparatus reference.


In [ ]:
import os, subprocess, json
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ROOT = Path("/kaggle/working")
REPOS = {
  "Wan2.2": ("https://github.com/abhinavk0006/Wan2.2.git", "43be1f5"),
  "image-to-video-pipeline": ("https://github.com/abhinavk0006/image-to-video-pipeline.git", "b5619f0"),
  "Edu-video-gen-dataset": ("https://github.com/abhinavk0006/Edu-video-gen-dataset.git", "53b5222")
}
for name, (url, commit) in REPOS.items():
    path = ROOT / name
    if not path.exists(): subprocess.run(["git", "clone", url, str(path)], check=True)
    fetch_ref = "feature/persistent-wan-worker" if name != "Edu-video-gen-dataset" else "feature/structured-reference-plans"
    subprocess.run(["git", "-C", str(path), "fetch", "origin", fetch_ref], check=True)
    subprocess.run(["git", "-C", str(path), "checkout", "--detach", commit], check=True)
image_matches = list(Path("/kaggle/input").rglob("chloride_salt_analysis.jpg"))
assert image_matches, "Add the dataset containing chloride_salt_analysis.jpg with Add Input"
IMAGE = image_matches[0]
print("Reference image:", IMAGE)
print("Pinned repositories:", REPOS)


In [ ]:
%pip install -q -r /kaggle/working/Edu-video-gen-dataset/requirements.txt imageio-ffmpeg
%pip install -q 'diffusers>=0.31,<0.33' 'transformers>=4.49,<=4.51.3' 'accelerate>=1.1.1' easydict safetensors ftfy huggingface_hub imageio


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 2), "GiB")
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2


## Build the six-segment experiment

Each segment is an independent shot from the real salt-analysis reference image. This is intentional: exact precipitate formation, litmus color, and dissolution are state changes, so chaining them from an unverified generated frame would propagate errors. The prompts are kept to one major procedure segment each.


In [ ]:
pipeline = ROOT / "image-to-video-pipeline"
fixture_dir = pipeline / "kaggle_experiments"
fixture_dir.mkdir(exist_ok=True)
base_negative = "blurry, distorted glassware, extra fingers, camera shake, warped test tubes, flickering, invented apparatus, text artifacts"
segments = [
  ("prepare_unknown_salt", 2.0, "A gloved student transfers a small amount of fine white unknown salt from the watch glass into one clean test tube in the wooden rack using a spatula. Keep the salt white and dry, the rack and four test tubes stable, and the camera fixed. Do not perform any reaction yet.", base_negative + ", colored powder, liquid, precipitate, reaction"),
  ("test_ammonium_with_naoh", 2.0, "Using the test tube containing white unknown salt, a dropper adds clear sodium hydroxide solution and a glass rod stirs gently. Damp red litmus paper is held at the mouth; it turns blue as invisible ammonia gas rises. Keep the apparatus stable and show one test only.", base_negative + ", litmus staying red, litmus turning other colors, colored solution, precipitate, flames"),
  ("nessler_ammonium_test", 2.0, "A dropper bottle of pale yellow Nessler reagent adds liquid to a fresh test tube containing unknown salt solution. A brown precipitate forms and becomes visible as the tube is gently swirled. Keep the brown precipitate localized in the test tube.", base_negative + ", no precipitate, white precipitate, blue precipitate, green precipitate, violent reaction"),
  ("silver_nitrate_chloride_test", 2.0, "A fresh test tube contains unknown salt solution. Dilute nitric acid is added first, followed by silver nitrate solution. A thick white curdy precipitate forms in the test tube and remains visible during a gentle swirl.", base_negative + ", no precipitate, yellow precipitate, brown precipitate, colored solution, violent reaction"),
  ("ammonia_dissolves_chloride_ppt", 2.0, "The test tube contains the thick white curdy precipitate from the silver nitrate chloride test. Dilute ammonia solution is added gradually and the white precipitate dissolves, leaving a clear colorless solution. Keep the tube and rack stable.", base_negative + ", precipitate remaining, colored solution, new precipitate, yellow precipitate, violent reaction"),
  ("final_salt_conclusion", 1.5, "A lab notebook beside the salt-analysis rack records the conclusion: ammonium ion confirmed by ammonia and Nessler tests; chloride ion confirmed by white silver chloride precipitate soluble in ammonia; unknown salt is ammonium chloride NH4Cl. Show the apparatus as a stable final overview. Do not rely on generated handwriting being readable.", base_negative + ", wrong conclusion, unrelated apparatus, dramatic reaction, camera movement")
]
structured = {
  "prepare_unknown_salt": {"state_before": "clean test tube and dry white unknown salt", "action": "transfer a small amount of salt into the test tube", "state_after": "white dry salt in the test tube", "visual_priority": "high", "continuity_required": False, "reference_policy": "state_keyframe"},
  "test_ammonium_with_naoh": {"state_before": "white unknown salt in the test tube", "action": "add sodium hydroxide and gently warm while holding damp red litmus at the mouth", "state_after": "litmus turns blue as ammonia is released", "visual_priority": "high", "continuity_required": False, "reference_policy": "state_keyframe"},
  "nessler_ammonium_test": {"state_before": "clear unknown salt solution in a fresh test tube", "action": "add Nessler reagent and gently swirl", "state_after": "localized brown precipitate", "visual_priority": "high", "continuity_required": False, "reference_policy": "state_keyframe"},
  "silver_nitrate_chloride_test": {"state_before": "clear acidified unknown salt solution", "action": "add silver nitrate and gently swirl", "state_after": "thick white curdy silver chloride precipitate", "visual_priority": "high", "continuity_required": False, "reference_policy": "state_keyframe"},
  "ammonia_dissolves_chloride_ppt": {"state_before": "white curdy silver chloride precipitate already visible in the test tube", "action": "add dilute ammonia gradually to the existing precipitate", "state_after": "clear colorless solution", "visual_priority": "high", "continuity_required": True, "reference_policy": "previous_state_keyframe", "reference_clip": "silver_nitrate_chloride_test"},
  "final_salt_conclusion": {"state_before": "completed salt-analysis apparatus", "action": "show the stable apparatus and conclusion card", "state_after": "stable final overview identifying ammonium chloride", "visual_priority": "normal", "continuity_required": False, "reference_policy": "state_keyframe"}
}
clips=[]
for name, duration, prompt, negative in segments:
    clips.append({"name": name, **structured[name], "duration_seconds": duration, "scene": {"scene": "High school chemistry laboratory with wooden test-tube rack, white unknown salt, reagent dropper bottles, glass rod, watch glass, and stable front-facing camera.", "camera": "Static front-angle educational shot", "lighting": "Bright consistent laboratory lighting"}, "prompt_bundle": {"motion_prompt": prompt, "negative_prompt": negative}, "handoff_mode": "final"})
fixture={"name":"Ammonium Chloride Salt Analysis", "description":"Six independent educational shots based on the supplied chloride salt-analysis procedure.", "subject":"Qualitative salt analysis", "educational_goal":"Demonstrate tests supporting NH4+ and Cl- in an unknown salt.", "knowledge_source":"user_supplied_salt_analysis_prompts", "clips":clips}
fixture_path=fixture_dir / "salt_analysis_NH4Cl_six_segments.json"
fixture_path.write_text(json.dumps(fixture, indent=2), encoding="utf-8")
print(fixture_path, "clips:", len(clips))


## Persistent-worker test

This notebook uses one long-lived Wan daemon for all six clips. It keeps T5/VAE infrastructure loaded and retains the last staged expert when possible; the opposite expert may still be swapped to stay within two T4 GPUs. Compare the total wall-clock time with the earlier native run (about 2h16m).


## Generate the full six-shot pilot

This uses the persistent Wan daemon, native area first, and the working 6/10 GiB placement. The daemon keeps shared model infrastructure loaded across clips. If native area OOMs, rerun with `--wan-max-area 345600`.


In [ ]:
%cd /kaggle/working/image-to-video-pipeline
import os
os.environ["IMAGE"] = str(IMAGE)
!python main.py --knowledge-file /kaggle/working/image-to-video-pipeline/kaggle_experiments/salt_analysis_NH4Cl_six_segments.json --initial-image $IMAGE --output-dir /kaggle/working/wan_salt_analysis_persistent_native --generator-script kaggle_wan_wrapper.py --generator-working-dir /kaggle/working/image-to-video-pipeline --wan-repo-dir /kaggle/working/Wan2.2 --wan-model-preset i2v-a14b --continuity-mode chain --generator-arg=--lightning --wan-max-area 345600 --wan-fallback-area 230400 --wan-fallback-area 172800 --generator-arg=--gpu0-memory-gib=5 --generator-arg=--gpu1-memory-gib=11 --generator-arg=--memory-telemetry


In [ ]:
from pathlib import Path
out=Path("/kaggle/working/wan_salt_analysis_persistent_native/ammonium_chloride_salt_analysis")
for path in sorted(out.rglob("*.mp4")):
    print(path, path.stat().st_size, "bytes")
print("Final:", out / "final_video.mp4")


## Reduced-area fallback

Run only if the native run OOMs or if you want a speed/VRAM comparison. This changes generation area, not the prompts or experiment sequence.


In [ ]:
%cd /kaggle/working/image-to-video-pipeline
import os
os.environ["IMAGE"] = str(IMAGE)
!python main.py --knowledge-file /kaggle/working/image-to-video-pipeline/kaggle_experiments/salt_analysis_NH4Cl_six_segments.json --initial-image $IMAGE --output-dir /kaggle/working/wan_salt_analysis_persistent_low_area --generator-script kaggle_wan_wrapper.py --generator-working-dir /kaggle/working/image-to-video-pipeline --wan-repo-dir /kaggle/working/Wan2.2 --wan-model-preset i2v-a14b --continuity-mode chain --wan-max-area 345600 --generator-arg=--lightning --wan-max-area 345600 --wan-fallback-area 230400 --wan-fallback-area 172800 --generator-arg=--gpu0-memory-gib=5 --generator-arg=--gpu1-memory-gib=11 --generator-arg=--memory-telemetry
